# TFT Beach Occupancy Prediction — Usage Guide

This notebook demonstrates how to use the `TFTService` to generate beach occupancy forecasts.
All data is **synthetic** — no database or webcam connection required.

---

## Architecture overview

```
TFTService.predict(webcam, days, since, model_set)
    │
    ├── _build_context()     ← historical snapshots + weather
    ├── _build_future()      ← future exogenous features
    ├── nf.predict()         ← NeuralForecast TFT inference
    └── _format_output()     ← per-hour prediction dicts
```

## Three models available

| Key  | Horizon | Days | Best for |
|------|---------|------|----------|
| `3d`  | H=36   | 1–3  | Highest accuracy, short-term |
| `10d` | H=120  | 1–10 | Weekly planning |
| `15d` | H=180  | 1–15 | Two-week outlook |

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import os
import sys
import json
import zoneinfo
from datetime import datetime, timedelta, date, timezone
from pathlib import Path
from types import SimpleNamespace
from unittest.mock import MagicMock, patch

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

_SPAIN_TZ = zoneinfo.ZoneInfo('Europe/Madrid')
HOUR_MIN, HOUR_MAX = 8, 20
HOURS_PER_DAY = HOUR_MAX - HOUR_MIN + 1

print('Imports OK')

## 1. Fake webcam + snapshots

In production these come from Django ORM (`WebCam`, `Snapshot` models).
Here we build lightweight stand-ins with the same attributes the service reads.

In [ ]:
# ── Fake beach / webcam ───────────────────────────────────────────────────────

fake_beach = SimpleNamespace(
    id=1,
    beach_name='Platja de Muro',
    coordenadas_geograficas='39.8028,3.1256',   # Mallorca north coast
)

fake_webcam = SimpleNamespace(
    camera_slug='platja-de-muro',
    camera_latitude=39.8028,
    camera_longitude=3.1256,
    max_crowd_count=80,          # P99 calibrated people count
    beach=fake_beach,
)

print(f'Webcam : {fake_webcam.camera_slug}')
print(f'Beach  : {fake_beach.beach_name}')
print(f'Coords : {fake_webcam.camera_latitude}, {fake_webcam.camera_longitude}')
print(f'Max cap: {fake_webcam.max_crowd_count} people')

In [ ]:
# ── Fake snapshots (48 hours of history — the model input_size) ───────────────
# Each snapshot = one hourly crowd count reading from the webcam.
# In production: Snapshot.objects.filter(webcam=cam).order_by('-ts')[:input_size*3]

np.random.seed(42)

CONTEXT_HOURS = 48   # how many history hours to feed the model
since_date = datetime(2025, 7, 15, 8, 0, tzinfo=_SPAIN_TZ)   # prediction start

def synthetic_crowd(ts, max_count=80):
    """Simulate realistic beach occupancy: seasonal + daily pattern + noise."""
    hour = ts.hour
    month = ts.month
    hour_effect  = np.exp(-((hour - 13.5) ** 2) / 18)          # peak at 13:30
    summer_boost = 0.4 + 0.6 * np.sin(np.pi * (month - 4) / 6) # peak Jul-Aug
    base = max_count * 0.7 * summer_boost * hour_effect
    noise = np.random.normal(0, max_count * 0.08)
    return max(0.0, base + noise)

# Build fake snapshot objects
FakeSnapshot = SimpleNamespace  # just a namespace with ts + predicted_crowd_count

snapshots = []
for h in range(CONTEXT_HOURS):
    ts = since_date - timedelta(hours=CONTEXT_HOURS - h)
    if HOUR_MIN <= ts.hour <= HOUR_MAX:
        snapshots.append(SimpleNamespace(
            ts=ts,
            predicted_crowd_count=synthetic_crowd(ts, fake_webcam.max_crowd_count)
        ))

print(f'{len(snapshots)} context snapshots')
print(f'From : {snapshots[0].ts}')
print(f'To   : {snapshots[-1].ts}')
print()
print('Sample (first 5):')
for s in snapshots[:5]:
    print(f'  {s.ts.strftime("%Y-%m-%d %H:%M")}  crowd={s.predicted_crowd_count:.1f}')

## 2. Weather data

The model uses two types of weather features:

| Type | Features | Notes |
|------|----------|-------|
| `futr_exog` | `om_temperature_2m` | Known for future timesteps (NWP forecast) |
| `hist_exog` | `om_shortwave_radiation`, `om_cloud_cover_low`, `om_vapour_pressure_deficit` | Historical only — model never depends on their future forecast accuracy |

In production `weather_module.get_for_period()` fetches from Open-Meteo API.
Here we synthesize plausible July values.

In [ ]:
# ── Fake weather dataframe ────────────────────────────────────────────────────
# Columns required by the service: ds_real, hour, + all om_* features.
# In production: weather_module.get_for_period(lat, lon, start, end)

WEATHER_COLS = [
    'om_temperature_2m',
    'om_shortwave_radiation',
    'om_cloud_cover_low',
    'om_vapour_pressure_deficit',
]

def fake_weather(start_dt, n_hours=120):
    rows = []
    for i in range(n_hours):
        ts = start_dt + timedelta(hours=i)
        h = ts.hour
        rows.append({
            'ds_real': ts.replace(tzinfo=None),
            'hour': h,
            'om_temperature_2m':          26 + 4 * np.sin(np.pi * h / 14) + np.random.normal(0, 0.5),
            'om_shortwave_radiation':     max(0, 600 * np.sin(np.pi * max(h - 6, 0) / 12) + np.random.normal(0, 30)),
            'om_cloud_cover_low':         max(0, min(100, 10 + np.random.normal(0, 8))),
            'om_vapour_pressure_deficit': max(0, 1.2 + 0.8 * np.sin(np.pi * h / 14) + np.random.normal(0, 0.1)),
        })
    return pd.DataFrame(rows)

weather_start = since_date - timedelta(days=5)
weather_df = fake_weather(weather_start, n_hours=5 * 24 + 16)  # past + future

print(f'Weather rows : {len(weather_df)}')
print(f'Date range   : {weather_df["ds_real"].min()} → {weather_df["ds_real"].max()}')
print()
print(weather_df[weather_df['hour'].between(8, 20)][WEATHER_COLS].describe().round(2))

## 3. Build the context dataframe

`_build_context()` is what the service calls internally.
It produces the NeuralForecast `df` input:
- `unique_id` = camera slug  
- `ds` = integer step index (not real timestamps — NeuralForecast requirement)  
- `y` = crowd count  
- all `futr_exog` + `hist_exog` columns merged from weather

In [ ]:
# ── Manually replicate _build_context() ──────────────────────────────────────

TEMPORAL_FEATURES = {
    'hour':        lambda ts: ts.hour,
    'day_of_week': lambda ts: ts.weekday(),
    'month':       lambda ts: ts.month,
    'is_weekend':  lambda ts: int(ts.weekday() >= 5),
    'is_summer':   lambda ts: int(ts.month in (6, 7, 8)),
}

FUTR_FEATURES = ['hour', 'day_of_week', 'month', 'is_weekend', 'is_summer', 'om_temperature_2m']
HIST_FEATURES = ['om_shortwave_radiation', 'om_cloud_cover_low', 'om_vapour_pressure_deficit']
ALL_FEATURES  = FUTR_FEATURES + HIST_FEATURES
INPUT_SIZE    = 48

weather_dt = weather_df.copy()
weather_dt['ds_real'] = pd.to_datetime(weather_dt['ds_real'])

rows = []
for s in snapshots:
    ts = s.ts.astimezone(_SPAIN_TZ).replace(tzinfo=None)
    row = {'ds_real': ts, 'y': float(s.predicted_crowd_count)}
    for col in ALL_FEATURES:
        if col in TEMPORAL_FEATURES:
            row[col] = TEMPORAL_FEATURES[col](ts)
    rows.append(row)

context_df = pd.DataFrame(rows)
context_df['unique_id'] = fake_webcam.camera_slug
context_df = context_df.sort_values('ds_real').tail(INPUT_SIZE).reset_index(drop=True)
context_df['ds'] = range(len(context_df))

# Merge weather features
for idx, row in context_df.iterrows():
    diffs = (weather_dt['ds_real'] - row['ds_real']).abs()
    nearest = diffs.idxmin()
    for col in HIST_FEATURES + ['om_temperature_2m']:
        context_df.loc[idx, col] = float(weather_dt.loc[nearest, col])

context_df = context_df.fillna(0)

print('context_df shape:', context_df.shape)
print()
print(context_df[['ds', 'ds_real', 'y'] + ALL_FEATURES[:4]].head(8).to_string(index=False))

## 4. Build the future exogenous dataframe

`_build_future()` builds the `futr_df` that NeuralForecast uses to look ahead.
It must contain exactly `H` rows (one per forecast step) starting right after the last context row.

In [ ]:
# ── Manually replicate _build_future() ───────────────────────────────────────

HORIZON_DAYS = 3
H = HORIZON_DAYS * HOURS_PER_DAY   # 36 steps for the 3d model

last_ds   = int(context_df['ds'].max())
last_real = context_df['ds_real'].iloc[-1]

future_weather = weather_dt[weather_dt['ds_real'] > last_real].head(H)

future_rows = []
for step in range(1, H + 1):
    future_ts = last_real + timedelta(hours=step)
    row = {'unique_id': fake_webcam.camera_slug, 'ds': last_ds + step}

    # Temporal features computed from timestamp
    for col in FUTR_FEATURES:
        if col in TEMPORAL_FEATURES:
            row[col] = TEMPORAL_FEATURES[col](future_ts)

    # Weather from NWP forecast
    if step <= len(future_weather):
        w = future_weather.iloc[step - 1]
        row['om_temperature_2m'] = float(w['om_temperature_2m'])
    else:
        row['om_temperature_2m'] = future_rows[-1]['om_temperature_2m'] if future_rows else 26.0

    future_rows.append(row)

futr_df = pd.DataFrame(future_rows)[['unique_id', 'ds'] + FUTR_FEATURES].fillna(0)

print(f'futr_df shape: {futr_df.shape}  (H={H} steps = {HORIZON_DAYS} days × {HOURS_PER_DAY} h/day)')
print()
print(futr_df.head(13).to_string(index=False))  # first day

## 5. Static features

Per-beach statistics computed from the training data y-series.
These are stored in `static_features.csv` inside each trained model directory.

In [ ]:
# ── Static features ───────────────────────────────────────────────────────────
# In production: loaded from apps/prediction/tft_models/{set_name}/tft_model_3d/static_features.csv
# Here we compute them from our synthetic context.

y = context_df['y']
static_df = pd.DataFrame([{
    'unique_id':    fake_webcam.camera_slug,
    'stat_mean_y':  float(y.mean()),
    'stat_cv':      float(y.std() / max(float(y.mean()), 1)),
}])

print('static_df:')
print(static_df.to_string(index=False))
print()
print('stat_mean_y  = mean historical occupancy (helps model calibrate scale per beach)')
print('stat_cv      = coefficient of variation (helps model understand volatility)')

## 6. Run inference with NeuralForecast

This is what `TFTService.predict()` calls after building the dataframes above:

```python
forecast = m['nf'].predict(
    df=context_df,
    static_df=static_df,
    futr_df=futr_df[['unique_id', 'ds'] + futr_features],
)
```

The output is a dataframe with columns `[unique_id, ds, TFT]`.

In [ ]:
# ── Simulate NeuralForecast output ────────────────────────────────────────────
# In production: nf.predict() returns this structure.
# We fake it here so the formatting step below works without a real model.

def simulate_tft_output(futr_df, webcam, noise_scale=0.08):
    """Fake TFT forecast with realistic daily pattern + some noise."""
    predictions = []
    for _, row in futr_df.iterrows():
        hour = int(row['hour'])
        month = int(row['month'])
        hour_effect  = np.exp(-((hour - 13.5) ** 2) / 18)
        summer_boost = 0.4 + 0.6 * np.sin(np.pi * (month - 4) / 6)
        temp_bonus   = max(0, (row['om_temperature_2m'] - 20) / 20)
        cc = webcam.max_crowd_count * 0.65 * summer_boost * hour_effect * (1 + temp_bonus)
        cc += np.random.normal(0, webcam.max_crowd_count * noise_scale)
        predictions.append(max(0.0, cc))
    df = futr_df[['unique_id', 'ds']].copy()
    df['TFT'] = predictions
    return df.set_index(['unique_id', 'ds'])

np.random.seed(7)
forecast_raw = simulate_tft_output(futr_df, fake_webcam)
forecast_raw = forecast_raw.reset_index()

print('NeuralForecast output shape:', forecast_raw.shape)
print()
print(forecast_raw.head(13).to_string(index=False))

## 7. Format output

`_format_output()` maps TFT steps back to real timestamps and builds
the per-hour prediction dicts with occupancy classification.

In [ ]:
# ── Format output (replicate _format_output) ─────────────────────────────────

OCCUPANCY_THRESHOLDS = [
    (0.75, 'HIGH'),
    (0.50, 'MEDIUM'),
    (0.25, 'LOW'),
    (0.0,  'VERY_LOW'),
]

def classify(crowd_count, max_cc):
    if not max_cc:
        return None
    r = crowd_count / max_cc
    for thresh, level in OCCUPANCY_THRESHOLDS:
        if r >= thresh:
            return level
    return 'VERY_LOW'

# Build prediction map: integer step -> crowd_count
step_to_cc = dict(zip(forecast_raw['ds'], forecast_raw['TFT'].clip(lower=0)))

# Map steps back to real timestamps
last_real_floored = last_real.replace(minute=0, second=0, microsecond=0)
if last_real_floored.hour >= HOUR_MAX:
    first_forecast_ts = (last_real_floored + timedelta(days=1)).replace(hour=HOUR_MIN, minute=0)
else:
    first_forecast_ts = last_real_floored + timedelta(hours=1)

pred_map = {}
ts = first_forecast_ts
for i, step in enumerate(sorted(step_to_cc)):
    pred_map[ts.strftime('%Y-%m-%dT%H:00')] = step_to_cc[step]
    ts += timedelta(hours=1)
    if ts.hour > HOUR_MAX:
        ts = (ts + timedelta(days=1)).replace(hour=HOUR_MIN, minute=0)

# Build final prediction list
start_day = first_forecast_ts.replace(hour=0, minute=0)
max_cc = fake_webcam.max_crowd_count
predictions = []

for day_offset in range(HORIZON_DAYS):
    current_day = start_day + timedelta(days=day_offset)
    for hour in range(24):
        t = current_day.replace(hour=hour)
        key = t.strftime('%Y-%m-%dT%H:00')
        if key in pred_map:
            cc = round(pred_map[key], 1)
            predictions.append({
                'timestamp':       t.isoformat(),
                'hour':            hour,
                'crowd_count':     cc,
                'available':       True,
                'occupancy_level': classify(cc, max_cc),
                'occupancy_ratio': round(cc / max_cc, 3),
            })
        else:
            predictions.append({
                'timestamp':       t.isoformat(),
                'hour':            hour,
                'crowd_count':     0,
                'available':       False,
                'occupancy_level': None,
                'occupancy_ratio': None,
            })

result = {
    'model':           '3d',
    'beach':           fake_beach.beach_name,
    'webcam':          fake_webcam.camera_slug,
    'horizon_days':    HORIZON_DAYS,
    'max_crowd_count': max_cc,
    'last_data':       last_real.isoformat(),
    'predictions':     predictions,
}

# Show daytime-only
daytime = [p for p in predictions if p['available']]
print(f'Total prediction slots : {len(predictions)}')
print(f'Daytime slots (8–20)   : {len(daytime)}')
print()
print('First day sample:')
print(f'{"Timestamp":<22} {"Count":>6} {"Ratio":>6}  Level')
print('-' * 50)
for p in daytime[:13]:
    print(f"{p['timestamp']:<22} {p['crowd_count']:>6.1f} {p['occupancy_ratio']:>6.3f}  {p['occupancy_level']}")

## 8. Using TFTService directly (production)

With Django set up and model checkpoints in `TFT_MODELS_DIR`, this is all you need:

In [ ]:
# ── Production usage (reference — requires Django + model files) ──────────────
'''
from apps.prediction.tft_service import TFTService
from apps.webcam.models import WebCam

service = TFTService()   # singleton — loads once, reused across requests
webcam  = WebCam.objects.select_related('beach').get(camera_slug='platja-de-muro')

# ── Basic: predict 3 days with best available model set ──
result = service.predict(webcam, days=3)

# ── Specific model set ──
result = service.predict(webcam, days=10, model_set='tft_train_2022_validate_2025_summer')

# ── Hindcast (past date, for evaluation) ──
from datetime import datetime, timezone
since = datetime(2025, 8, 1, tzinfo=timezone.utc)
result = service.predict(webcam, days=3, since=since)

# ── Cascading fill: 3d → 10d → 15d for 15-day forecast ──
result = service.predict_mixed(webcam, days=15)
result = service.predict_mixed(webcam, days=15, model_set='tft_train_2022_validate_2025_summer')

# ── List available model sets ──
print(service.list_model_sets())

# ── Result structure ──
result.keys()
# dict_keys(['model', 'model_set', 'beach', 'webcam', 'horizon_days',
#            'max_crowd_count', 'last_data', 'futr_features', 'hist_features',
#            'feature_importance', 'predictions'])

# ── Each prediction slot ──
for p in result['predictions']:
    if p['available']:
        print(p['timestamp'], p['crowd_count'], p['occupancy_level'])
'''
print('Reference cell — run in Django context only')

## 9. Visualise the forecast

In [ ]:
# ── Plot 1: Forecast bar chart (per-hour, colored by occupancy level) ─────────

LEVEL_COLORS = {
    'HIGH':     '#dc2626',
    'MEDIUM':   '#ea580c',
    'LOW':      '#16a34a',
    'VERY_LOW': '#2563eb',
    None:       '#e2e8f0',
}

daytime = [p for p in predictions if p['available']]
x_labels = [p['timestamp'][5:16].replace('T', ' ') for p in daytime]  # MM-DD HH
counts   = [p['crowd_count'] for p in daytime]
colors   = [LEVEL_COLORS[p['occupancy_level']] for p in daytime]

fig, axes = plt.subplots(2, 1, figsize=(14, 7))

# Bar chart
ax = axes[0]
bars = ax.bar(range(len(daytime)), counts, color=colors, width=0.8)
ax.axhline(max_cc * 0.75, color='#dc2626', lw=0.8, ls='--', alpha=0.6, label='HIGH (75%)')
ax.axhline(max_cc * 0.50, color='#ea580c', lw=0.8, ls='--', alpha=0.6, label='MEDIUM (50%)')
ax.axhline(max_cc * 0.25, color='#16a34a', lw=0.8, ls='--', alpha=0.6, label='LOW (25%)')
ax.set_xticks(range(len(daytime)))
ax.set_xticklabels(x_labels, rotation=45, ha='right', fontsize=7)
ax.set_ylabel('Crowd count')
ax.set_title(f'3-day forecast — {fake_beach.beach_name}  (model: 3d, max_cap={max_cc})')
ax.legend(fontsize=8)
ax.grid(axis='y', alpha=0.3)

# Occupancy ratio line
ax2 = axes[1]
ratios = [p['occupancy_ratio'] for p in daytime]
ax2.plot(range(len(daytime)), ratios, color='#1e3a5f', lw=1.4)
ax2.fill_between(range(len(daytime)), ratios, alpha=0.15, color='#1e3a5f')
ax2.axhline(0.75, color='#dc2626', lw=0.8, ls='--', alpha=0.6)
ax2.axhline(0.50, color='#ea580c', lw=0.8, ls='--', alpha=0.6)
ax2.axhline(0.25, color='#16a34a', lw=0.8, ls='--', alpha=0.6)
ax2.set_xticks(range(len(daytime)))
ax2.set_xticklabels(x_labels, rotation=45, ha='right', fontsize=7)
ax2.set_ylabel('Occupancy ratio')
ax2.set_ylim(0, 1.05)
ax2.set_title('Occupancy ratio (crowd_count / max_crowd_count)')
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('tft_forecast_demo.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Plot 2: Context (actual history) + forecast ───────────────────────────────

ctx_times  = [s.ts.replace(tzinfo=None) for s in snapshots if HOUR_MIN <= s.ts.hour <= HOUR_MAX]
ctx_counts = [s.predicted_crowd_count for s in snapshots if HOUR_MIN <= s.ts.hour <= HOUR_MAX]

fcast_times  = [pd.Timestamp(p['timestamp']) for p in daytime]
fcast_counts = [p['crowd_count'] for p in daytime]

fig, ax = plt.subplots(figsize=(14, 4.5))
ax.plot(ctx_times, ctx_counts, color='#333', lw=1.4, label='Actual (context)')
ax.plot(fcast_times, fcast_counts, color='#2563eb', lw=1.4, ls='--', label='Forecast (3d model)')
ax.axvline(pd.Timestamp(last_real), color='#888', lw=1, ls=':', alpha=0.8)
ax.text(pd.Timestamp(last_real), ax.get_ylim()[1] * 0.95, '  last data', fontsize=8, color='#888')
ax.fill_between(fcast_times, fcast_counts, alpha=0.1, color='#2563eb')
ax.set_ylabel('Crowd count')
ax.set_title(f'Context + 3-day forecast — {fake_beach.beach_name}')
ax.legend(fontsize=9)
ax.grid(alpha=0.3)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%d %b %H:00'))
ax.xaxis.set_major_locator(mdates.HourLocator(byhour=[8, 14, 20]))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha='right', fontsize=8)
plt.tight_layout()
plt.savefig('tft_context_forecast_demo.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Plot 3: Feature importance (from config.json in production) ───────────────
# In production: result['feature_importance'] comes from the trained model.
# Here we use the known optimal feature set from the thesis training experiments.

feature_importance = {
    'om_shortwave_radiation':      0.28,
    'hour':                        0.22,
    'om_temperature_2m':           0.18,
    'om_cloud_cover_low':          0.12,
    'om_vapour_pressure_deficit':  0.10,
    'month':                       0.05,
    'is_weekend':                  0.03,
    'is_summer':                   0.01,
    'day_of_week':                 0.01,
}

features = list(feature_importance.keys())
values   = list(feature_importance.values())
colors_fi = ['#2563eb' if f.startswith('om_') else '#64748b' for f in features]

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.barh(features[::-1], values[::-1], color=colors_fi[::-1])
ax.set_xlabel('VSN importance (normalized)')
ax.set_title('Feature importance — TFT 3d model')
ax.grid(axis='x', alpha=0.3)

from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(color='#2563eb', label='Weather (om_*)'),
    Patch(color='#64748b', label='Temporal'),
], fontsize=9)
plt.tight_layout()
plt.savefig('tft_feature_importance_demo.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Key parameters reference

| Parameter | Value | Notes |
|---|---|---|
| `input_size` | 48 | Context steps fed to model (4 days × 13h) |
| `hidden_size` | 64 | TFT hidden layer size |
| `n_head` | 4 | Multi-head attention heads |
| `dropout` | 0.05 | Regularization |
| `lr` | 0.001 | Adam learning rate |
| `scaler_type` | minmax | Per-series normalization |
| `loss` | MAE | Training objective |
| `HOUR_MIN/MAX` | 8 / 20 | Filtered daytime window (13h/day) |
| `futr_exog` | `[hour, day_of_week, month, is_weekend, is_summer, om_temperature_2m]` | Known at forecast time |
| `hist_exog` | `[om_shortwave_radiation, om_cloud_cover_low, om_vapour_pressure_deficit]` | Historical only |
| `stat_exog` | `[stat_mean_y, stat_cv]` | Per-beach static |

## 11. Prediction result schema

```json
{
  "model": "3d",
  "model_set": "tft_train_2022_validate_2025_summer",
  "beach": "Platja de Muro",
  "webcam": "platja-de-muro",
  "horizon_days": 3,
  "horizon_hours": 36,
  "max_crowd_count": 80,
  "last_data": "2025-07-14T19:00:00",
  "futr_features": ["hour", "day_of_week", "month", "is_weekend", "is_summer", "om_temperature_2m"],
  "hist_features": ["om_shortwave_radiation", "om_cloud_cover_low", "om_vapour_pressure_deficit"],
  "feature_importance": { "om_shortwave_radiation": 0.28, ... },
  "predictions": [
    {
      "timestamp": "2025-07-15T08:00:00",
      "hour": 8,
      "crowd_count": 12.4,
      "available": true,
      "occupancy_level": "VERY_LOW",
      "occupancy_ratio": 0.155,
      "temperature": 24.1,
      "features": { "om_temperature_2m": 24.1, "om_shortwave_radiation": 320.0, ... }
    },
    {
      "timestamp": "2025-07-15T00:00:00",
      "hour": 0,
      "crowd_count": 0,
      "available": false,
      "occupancy_level": null,
      "occupancy_ratio": null
    }
  ]
}
```